## Init containers

Sometimes a pod must do *something* before the main container starts — fetch a config file, run a migration, wait for a dependency. That's what **init containers** are for.

Init containers run **one at a time, in order, to completion, before any main container starts.** If one fails, the pod's `restartPolicy` decides: `Always`/`OnFailure` retry; `Never` fails the pod.

```yaml
spec:
  initContainers:
    - name: prepare-content
      image: busybox:1.36
      command: ["sh", "-c", "echo hello > /shared/index.html"]
      volumeMounts:
        - { name: html, mountPath: /shared }
  containers:
    - name: app
      image: nginx:alpine
      volumeMounts:
        - { name: html, mountPath: /usr/share/nginx/html }
  volumes:
    - name: html
      emptyDir: {}
```

The sequence: pod scheduled → `prepare-content` writes into the shared `html` volume → it exits `0` → nginx starts and serves the file the init container wrote.

Internalise about init containers:

- **Same spec as main containers** — same image, env, and volume-mount fields. Anything a main container can do, an init container can.
- **Sequential** — `initContainers` is a list processed top-down; order matters.
- **They share the pod's volumes and network** — that's how they hand work off (write to a shared `emptyDir`).
- **Resources aren't simply summed** — the pod's effective request is the max init container against the sum of main containers (notebook 07).
- **`Initialized` flips to `True`** only after the last init container succeeds.

On our map this is the **initContainers** chip inside the **Pod** box — the run-once preamble the kubelet completes before the real workload begins.